# Prediction, multiple regression and F-tests in Python: practice notebook

**Causal and Predictive Analytics · Asynchronous module** · Simon Business School · Prof. Tianli Xia

All the Python from the module's web page, plus short exercises (cells marked `# YOUR CODE HERE`). Three small data sets:

| File | What it is |
|---|---|
| `hprices2.csv` | 88 houses: `price` (in $1,000s), size `sqrft`, `lotsize` (square feet), `bdrms` |
| `rfj_small.csv` | orange juice in 218 store-weeks: Tropicana's quantity `q1` (thousands of 64-ounce cartons) and the prices per carton `p1` (Tropicana), `p2` (Minute Maid), `p3` (private label) |
| `suv.csv` | 70 SUV models: market share `mshare`, `Invoice` price and `MSRP` (in $1,000s) |

**In Colab:** *File → Save a copy in Drive* first, so your changes are kept. Run cells with **Shift+Enter**.

## 0. Setup and data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from scipy import stats

In [ ]:
from pathlib import Path

SITE = "https://tianlixia.com/causal-analytics/GBA472/A5/"   # the module's web address; filled in when the notebook is published, so the data load in Colab


def load(name):
    """Read a data file from the folder next to this notebook, or from the module's website."""
    for folder in [Path("data"), Path("web/data"), Path("../web/data")]:
        if (folder / name).exists():
            return pd.read_csv(folder / name)
    return pd.read_csv(SITE + "data/" + name)


housedata = load("hprices2.csv")
ojdata = load("rfj_small.csv")
suv = load("suv.csv")
print(housedata.shape, ojdata.shape, suv.shape)

## 1. Prediction

### 1.1 The regression, and one prediction by hand

In [ ]:
est = smf.ols("price ~ sqrft", data=housedata).fit()
print(est.summary().tables[1])

# a 2,500 square foot house: plug its size into the regression line
print(11.2041 + 0.1402 * 2500)

### 1.2 `get_prediction`: the prediction and both intervals

Put the X values in a data frame whose column names match the formula. `alpha` is one minus the confidence level.

* `mean`: the point prediction; `mean_se`: its standard error
* `mean_ci_lower`, `mean_ci_upper`: the **confidence interval** for the average of all houses of that size
* `obs_ci_lower`, `obs_ci_upper`: the **prediction interval** for one house of that size

In [ ]:
d = {"sqrft": [1500, 2000, 2500, 3000, 3500, 4000]}
df = pd.DataFrame(data=d)

predictions = est.get_prediction(df)
print(predictions.summary_frame(alpha=0.05).round(1))     # alpha = 0.05: 95% intervals

**Exercise 1.** (a) Get **90%** intervals for the same six sizes. Are they wider or narrower? (b) A house of 2,200 square feet is on the market for $420,000. Is that price inside the 95% prediction interval for a house of this size? Inside the 95% confidence interval for the average? What do you conclude?

In [ ]:
# YOUR CODE HERE


### 1.3 With robust standard errors

The predictions are the same. Their standard errors, and so the intervals, change.

In [ ]:
est_r = smf.ols("price ~ sqrft", data=housedata).fit(cov_type="HC1")      # heteroskedasticity-robust
print(est_r.get_prediction(df).summary_frame(alpha=0.05).round(1))

### 1.4 What a larger sample changes

A lazy experiment: stack 100 copies of the data, as if each house stood for 100 houses exactly like it. **Never do this with real data.** It only shows what the sample size does to the two intervals.

In [ ]:
housedata_bigger = pd.concat([housedata] * 100, ignore_index=True)       # 8,800 rows
est_b = smf.ols("price ~ sqrft", data=housedata_bigger).fit()
print(est_b.summary().tables[1])
print(est_b.get_prediction(df).summary_frame(alpha=0.05).round(1))

**Exercise 2.** Repeat the experiment with **4** copies instead of 100. By what factor does the width of the confidence interval at 2,500 square feet fall? And the width of the prediction interval? What factor did you expect?

In [ ]:
# YOUR CODE HERE


## 2. Multiple regression

### 2.1 More X's: add them with `+`

In [ ]:
# formula: response ~ predictor1 + predictor2 + predictor3 + ...
est = smf.ols("price ~ sqrft + lotsize + bdrms", data=housedata).fit()
print(est.summary())

### 2.2 Why is the number of bedrooms not significant?

Most houses have three or four bedrooms: with so little variation in an X, its coefficient is imprecise.

In [ ]:
print(housedata.bdrms.value_counts().sort_index())

housedata.bdrms.hist(bins=range(2, 9), align="left", rwidth=0.8)
plt.xlabel("bedrooms"); plt.ylabel("houses")
plt.show()

### 2.3 Robust standard errors and predictions, with several X's

In [ ]:
est_r = smf.ols("price ~ sqrft + lotsize + bdrms", data=housedata).fit(cov_type="HC1")
print(est_r.summary().tables[1])

# predictions: now every X needs a value. Two houses:
d = {"sqrft": [2000, 3000], "lotsize": [4000, 5000], "bdrms": [3, 4]}
df2 = pd.DataFrame(data=d)
print(est.get_prediction(df2).summary_frame(alpha=0.05).round(1))

**Exercise 3.** Predict the price of a house with 2,400 square feet, a lot of 9,000 square feet and 4 bedrooms, with its 95% prediction interval. Then give it a fifth bedroom, with everything else unchanged. By how much does the prediction change, and where have you seen that number?

In [ ]:
# YOUR CODE HERE


### 2.4 Orange juice: own price, then the rivals' prices

In [ ]:
est1 = smf.ols("q1 ~ p1", data=ojdata).fit()
est3 = smf.ols("q1 ~ p1 + p2 + p3", data=ojdata).fit()
print(est1.summary().tables[1])
print(est3.summary().tables[1])

The own-price coefficient moves from −80.9 to −85.0 when the rivals' prices come in. The three prices are positively correlated, and the rivals' prices raise Tropicana's sales: omitted variable bias.

In [ ]:
print(ojdata[["p1", "p2", "p3"]].corr().round(3))

**Exercise 4.** Check the omitted-variable-bias formula with numbers: the coefficient on `p1` in the short regression equals the one in the long regression, plus (coefficient on `p2`) × (slope of `p2` on `p1`), plus (coefficient on `p3`) × (slope of `p3` on `p1`).

In [ ]:
# YOUR CODE HERE


### 2.5 Perfect collinearity: square feet and square meters

Python does not stop. It reports one of infinitely many ways to split the effect between the two variables, and warns about the smallest eigenvalue under the table. The numbers themselves can differ between versions of Python; the warning, and the two identical t statistics, are what to look for. The fix is to keep one of the two.

In [ ]:
housedata["sqrmt"] = 1 / 10.764 * housedata["sqrft"]
print(housedata[["sqrft", "sqrmt"]].corr())

est_pc = smf.ols("price ~ sqrft + sqrmt", data=housedata).fit()
print(est_pc.summary())

### 2.6 Near-perfect collinearity: two prices of the same car

In [ ]:
print(suv[["model", "mshare", "Invoice", "MSRP"]].head())

est_suv = smf.ols("mshare ~ Invoice + MSRP", data=suv).fit()
print(est_suv.summary())

In [ ]:
print(suv[["Invoice", "MSRP"]].corr().round(4))

est_msrp = smf.ols("mshare ~ MSRP", data=suv).fit()
print(est_msrp.summary().tables[1])

**Exercise 5.** Keep `Invoice` instead of `MSRP`. Is the conclusion the same?

In [ ]:
# YOUR CODE HERE


## 3. Joint hypotheses: F-tests

$$F=\frac{(R^2_U-R^2_R)/q}{(1-R^2_U)/(n-p-1)}$$

$R^2_U$: the unrestricted model; $R^2_R$: the model with the null imposed; $q$: the number of restrictions; $n-p-1$: the residual degrees of freedom of the unrestricted model.

### 3.1 Do lot size and bedrooms matter, jointly? By hand

In [ ]:
modelUR = smf.ols("price ~ sqrft + lotsize + bdrms", data=housedata).fit()       # unrestricted
modelR = smf.ols("price ~ sqrft", data=housedata).fit()                          # restricted: lotsize = bdrms = 0

q, df_resid = 2, modelUR.df_resid
F = ((modelUR.rsquared - modelR.rsquared) / q) / ((1 - modelUR.rsquared) / df_resid)
print("R2 unrestricted", round(modelUR.rsquared, 4), " R2 restricted", round(modelR.rsquared, 4))
print("F =", round(F, 2), " p-value =", round(stats.f.sf(F, q, df_resid), 4))

### 3.2 The same test in one line: `f_test`

In [ ]:
print(modelUR.f_test("lotsize = 0, bdrms = 0"))

**Exercise 6.** (a) Reproduce the F-statistic of the regression output (all three slopes are zero) with the formula. Which $R^2_R$ do you need? (b) Test the single restriction `bdrms = 0` with `f_test` and compare the result with the t statistic of `bdrms`.

In [ ]:
# YOUR CODE HERE


### 3.3 Orange juice: are the two rivals equally strong competitors?

$H_0:\beta_2=\beta_3$. By hand, the restricted model has one coefficient for `p2 + p3`.

In [ ]:
ojUR = smf.ols("q1 ~ p1 + p2 + p3", data=ojdata).fit()

ojdata["p23"] = ojdata["p2"] + ojdata["p3"]                 # the new variable
ojR = smf.ols("q1 ~ p1 + p23", data=ojdata).fit()

F = ((ojUR.rsquared - ojR.rsquared) / 1) / ((1 - ojUR.rsquared) / ojUR.df_resid)
print("R2 unrestricted", round(ojUR.rsquared, 4), " R2 restricted", round(ojR.rsquared, 4))
print("F =", round(F, 3), " p-value =", round(stats.f.sf(F, 1, ojUR.df_resid), 3))

In [ ]:
print(ojUR.f_test("p2 - p3 = 0"))          # the same test: "p2 = p3"

**Exercise 7.** Test whether the two rivals' prices matter at all for Tropicana's sales: $H_0:\beta_2=\beta_3=0$. Do it by hand and with `f_test`.

In [ ]:
# YOUR CODE HERE


---
*That's all. Next: comparisons over time, with difference-in-differences and the synthetic control method.*